## ML Climate Zones Tutorial - Training in PyTorch Lightning
* Author: Stephen Haddad
* Affiliation: UK Met Office
* History: 1.0
* Last update: 2026-02-10
* © British Crown Copyright 2017-2065, Met Office. Please see LICENSE.md for license details.

## Overview of the broad topic covered

In this notebook, we will use the previously prepared tabular dataset to predict climate zones. We will train on different eras to see how well the results generalise with climate change. 

### Prerequisites 
- Same as previous  notebooks
- Have completed the training pipeline, inference and evaluation notebooks.


### Learning outcomes from completing the notebook

- Understand how to build a pipeline using PyTorch
- Understand the core PyTorch concepts and classes
- Understand how to manage experiments using ML Flow

## Tutorial - Creating a machine learning pipe

Further Reading
* [PyTorch Docs](https://pytorch.org/)

## Setup
First we start by loading the data we have prepared previously, and other set up elements

### Environment 
This notebook uses the environment defined in this repository in the [environments/requirements_pytorch.yaml](../../../environments/requirements_pytorch.yaml) conda file.  

If you are running this notebook on Met Office IT, please follow the [guidance on using conda at the Met Office](https://wwwspice/~avd/sci/software_stack/conda_initial_how_to.html). For other platforms please consult the relevant platform specific guidance on using conda on that platform where it exists, or [general conda documentation](https://www.anaconda.com/docs/getting-started/miniconda/install/overview) for getting started.

### Imports

In [1]:
import pathlib
import os
import datetime
import json

In [2]:
import numpy 
import pandas

In [3]:
import matplotlib
import matplotlib.pyplot

In [4]:
import sklearn
import sklearn.preprocessing
import sklearn.tree


In [5]:
import mlflow

/gws/ssde/j25b/mohc_shared/users/shaddad/venv/dscop_torch_nb_gpu/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import torch
import lightning

## Load and prepare data 
We will now load the dataset and do the usual data prep steps, like train/test split and normalisation.


#### Dataset parameters

In [8]:
with open ('config.json','r') as tutorial_config:
    tutorial_config = json.load(tutorial_config)
tutorial_config

{'platform': 'mo_linux',
 'default_dirs': {'mo_linux': '/data/users/dscop/ml_tutorial/',
  'jasmin': '/gws/ssde/j25a/mmh_storage/ai4c_data/'},
 'random_seed': 12345,
 'climate_subgroups': {'non-land': 'None',
  'Af': 'Tropical, rainforest',
  'Am': 'Tropical, monsoon',
  'Aw': 'Tropical, savannah',
  'BWh': 'Arid, desert, hot',
  'BWk': 'Arid, desert, cold',
  'BSh': 'Arid, steppe, hot',
  'BSk': 'Arid, steppe, cold',
  'Csa': 'Temperate, dry summer, hot summer',
  'Csb': 'Temperate, dry summer, warm summer',
  'Csc': 'Temperate, dry summer, cold summer',
  'Cwa': 'Temperate, dry winter, hot summer',
  'Cwb': 'Temperate, dry winter, warm summer',
  'Cwc': 'Temperate, dry winter, cold summer',
  'Cfa': 'Temperate, no dry season, hot summer',
  'Cfb': 'Temperate, no dry season, warm summer',
  'Cfc': 'Temperate, no dry season, cold summer',
  'Dsa': 'Cold, dry summer, hot summer',
  'Dsb': 'Cold, dry summer, warm summer',
  'Dsc': 'Cold, dry summer, cold summer',
  'Dsd': 'Cold, dry summ

In [9]:
def get_platform_dir(select_platform, config):
    try:
        root_path = pathlib.Path(config['default_dirs'][select_platform]) / 'climate_zones'
    except KeyError:
        root_path = pathlib.Path(os.environ['HOME']) / 'climate_zones'
    return root_path

In [21]:
current_platform = 'jasmin'

In [22]:
current_platform

'jasmin'

In [23]:
root_data_dir = get_platform_dir(current_platform, tutorial_config)

print(root_data_dir.is_dir())
root_data_dir

True


PosixPath('/gws/ssde/j25a/mmh_storage/ai4c_data/climate_zones')

In [24]:
ml_ready_dir = root_data_dir / 'ml_ready'
print(ml_ready_dir.is_dir())
ml_ready_dir

True


PosixPath('/gws/ssde/j25a/mmh_storage/ai4c_data/climate_zones/ml_ready')

In [14]:
resolutions_dict = {float(k1): v1 for k1,v1 in tutorial_config['resolutions_names'].items()}

dataset_prefix_dict = tutorial_config['dataset_prefix']

format_str = 'nc'
historic_scenario_str = 'historic'

future_scenario_list = tutorial_config['future_scenarios']
historic_scenario_list = tutorial_config['historic_scenarios']

time_periods = { 
    (1901,1930): historic_scenario_list, 
    (1931,1960): historic_scenario_list,
    (1961,1990): historic_scenario_list,
    (1991,2020): historic_scenario_list,
    (2041,2070): future_scenario_list,
    (2071,2099): future_scenario_list,
}


In [15]:
fname_template = tutorial_config['fname_template']
time_dir_template = tutorial_config['time_dir_template']
ml_ready_fname_template = tutorial_config['csv_out_template']

#### Load data for training

In [16]:
current_res = 1.0

In [25]:
mlready_data_path = ml_ready_dir / ml_ready_fname_template.format(resolution=resolutions_dict[current_res])
print(mlready_data_path.is_file())
mlready_data_path

True


PosixPath('/gws/ssde/j25a/mmh_storage/ai4c_data/climate_zones/ml_ready/climate_zones_1p0.csv')

In [26]:
zones_df = pandas.read_csv(mlready_data_path)

We have two options for the target. We have the full 30 class climate subgroups of the Koppen-Geiger classification data from the original dataset. We also have the processed five class climate group target, which presents an easier target for our classification algorithm to predict.

#### Selecting features

In [27]:
predictors_dict = {
    'precip_mean': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'mean' in c1],
    'precip_std': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'std' in c1],
    'temp_mean': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'mean' in c1],
    'temp_std': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'std' in c1],
}


In [28]:
target_var = 'climate_group' # 5 classes
# target_var = 'climate_subgroup # 30 classes


In [29]:
predictors = predictors_dict['precip_mean'] + predictors_dict['temp_mean']
predictors

['precipitation_1.0_mean',
 'precipitation_2.0_mean',
 'precipitation_3.0_mean',
 'precipitation_4.0_mean',
 'precipitation_5.0_mean',
 'precipitation_6.0_mean',
 'precipitation_7.0_mean',
 'precipitation_8.0_mean',
 'precipitation_9.0_mean',
 'precipitation_10.0_mean',
 'precipitation_11.0_mean',
 'precipitation_12.0_mean',
 'air_temperature_1.0_mean',
 'air_temperature_2.0_mean',
 'air_temperature_3.0_mean',
 'air_temperature_4.0_mean',
 'air_temperature_5.0_mean',
 'air_temperature_6.0_mean',
 'air_temperature_7.0_mean',
 'air_temperature_8.0_mean',
 'air_temperature_9.0_mean',
 'air_temperature_10.0_mean',
 'air_temperature_11.0_mean',
 'air_temperature_12.0_mean']

#### Train/test split

In [30]:
random_seed = tutorial_config['random_seed']

In [31]:
test_frac = 0.1
val_frac = 0.1
val_frac_sub = (val_frac / (1.0-test_frac) )

In [32]:
test_df = zones_df.groupby(['period_start','scenario']).sample(frac=test_frac, random_state=random_seed)
remain_df = zones_df.drop(test_df.index)

In [33]:
val_df = remain_df.groupby(['period_start','scenario']).sample(frac=val_frac_sub, random_state=random_seed)
train_df = remain_df.drop(val_df.index)


## Using PyTorch

From this point in the tutorial, we diverge from what was done in the ml training pipeline tutorial as instead of setting up and training our model in scikit-learn, we're going to use a more sophisticated machine learning library called pytorch. This gives us more more control over how we implement our neural network and gives us much more power to use advanced architectures, losss functions and distributed computing techniques.

When using PyTorch, the main difference is that we will specify the details of the neural network architecture and training loop much more explictly. This means we can customise and optimise these details for the particular problem. Key elements of a PyTorch training pipeline, compared to what was previous described are as follows:
1. **Data Loading and Cleaning** - This is usually done through a PyTorch Dataset class. In addition, you create a Dataset Loader object, which has the responsibility for iterating through the data in the dataset, including shuffling data between epochs where appropriate.</p>
2. **Feature Engineering** - Same as before, but may be a part of the dataset class.</p>
3. **Train/Test split** - Same method as before. Usually different dataset objects will represent the train, validate and test sets.</p>
4. **Data Preparation** - Same as before, but code may be structured differently such that the normalisation and scaling happening inside the PyTorch dataset class.</p>
5. **Algorithm Setup** - Usually you create a class representing the model architecture. You then also specify key hyperparameters such as the optimiser for training the weights, the learning rate, etc. </p>
6. **Algorithm Training** - The elements of the training loop are described more explictly in PyTorch typically with a explicit loop for iterating through batches and an outer loop for iterating through batches.</p>
7. **Inference** - The model object is used as a callable object for producing. </p>
8. **Evaluation** - Evaluation is the same as before. </p>
9. **Interpretability and Explainability** - As model architectures become more complex, it becomes more difficult to explain or interpret the results, this is an active area of research. </p>
10. **Model Storage** - Pytorch has a more sophisticated mechanism for [saving and loading models](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html). </p>


### Check GPU availability

As a part of this notebook, we will now be training our models using the GPUs on JASMIN. There are some additional steps required for this purpose, such as moving the data and model onto the GPU memory for processing. ML frameworks are especially helpful for this in abtracting away many of the details of this into a few commands.

In this cell, we check whether there is a GPU to use. [CUDA](https://en.wikipedia.org/wiki/CUDA) is the underlying software layer that interfaces to nvidia GPUs. This check allows the notebook to seamlessly work either on a gpu if one is available or to do processing on a cpu when a gpu is not available.



In [34]:
# Autodetect GPU and use if possible
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

#### Training hyperparameters

In [35]:
training_params = {
    'batch_size': 16,
    'num_epochs': 2,
    'learning_rate': 0.001,
    'loss': 'CrossEntropyLoss',
    'criterion': 'CrossEntropyLoss',
    'optimizer': 'Adam',
}

### Define a data loader

Pytorch defines the interface between the dataset and machine learning through a base class (`torch.utils.data.Dataset), which follow [pythonic paradigms](https://realpython.com/ref/glossary/pythonic/) for software architecture. The implementation of the class hides away the details of the data being used, so that the dataset can be used in the generic pytorch architecture. There is also then a data loader which is an [iterator](https://www.w3schools.com/python/python_iterators.asp) on the dataset, enabling pytorch to progress through all the data during training. Ultimately the aim of this data architecture is to present the data in the correct [pytorch tensor format](https://docs.pytorch.org/docs/stable/tensors.html) expected by the neural network for training purposes.

#### Key Concepts:
- **Pytorch Dataset**: Handles loading and preparing the data for use with pytorch. Implements key [python built-in methods](), including:</p>
  - [`__init__`](https://docs.python.org/3/reference/datamodel.html#object.__init__) creates the dataset object, and typically loads and transforms the data ready for use, or in a lazy loading paradigm, define the task pipeline for loading the data upon request.
  - [`__len__`](https://docs.python.org/3/reference/datamodel.html#object.__len__) specifies how many data point there are.
  - [`__getitem__`](https://docs.python.org/3/reference/datamodel.html#object.__getitem__) return the item for a particular index.</p>
- [**Pytorch Data Loader**](https://docs.pytorch.org/docs/stable/data.html): Interfaces between the ML algorithm being trained and the dataset, selecting mini batches of data to use with each iteration of the gradient descent with back propogation used for training the neural network. Key parameters include:</p>
  - **dataset** - The dataset to load the data from (as described above).
  - [**batch size**](https://www.geeksforgeeks.org/deep-learning/batch-size-in-neural-network/) - How many samples to use in each mini batch during training.
  - [**shuffle**](https://stats.stackexchange.com/questions/245502/why-should-we-shuffle-data-while-training-a-neural-network) - Whether to shuffle the order of data use during each of the epochs. Data shuffling improves the generalisation of what is learned by the neural network.


Further reading
- [Intro to datasets and data loader - pytorch docs](https://docs.pytorch.org/tutorials/beginner/data_loading_tutorial.html)
- [Tutorial on using CSV data with pytorch data architecture - Machine Learning Mastery](https://machinelearningmastery.com/converting-pandas-dataframes-to-pytorch-dataloaders-for-custom-deep-learning-model-training/)
- [Converting pnadas dataframe to the pytorch](https://www.geeksforgeeks.org/deep-learning/converting-a-pandas-dataframe-to-a-pytorch-tensor/)
- [Encoding target data for a neural network using LabelBinarizer - scikit learn](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelBinarizer.html)

### Implementation details for this tutorial

In [36]:
class ClimateZonesDataset(torch.utils.data.Dataset):
    """
    Inspired by this tutorial:
    https://machinelearningmastery.com/converting-pandas-dataframes-to-pytorch-dataloaders-for-custom-deep-learning-model-training/
    """
    def __init__(self, df_ml, predictor_features, target_feature, device, stats_dict=None):
        self._df_ml = df_ml.reset_index().drop(['index'],axis='columns')
        self._device = device
        self.input_scaler = sklearn.preprocessing.StandardScaler()
        if stats_dict is None:
            self.input_scaler.fit(self._df_ml[predictor_features])
        else:
            self.input_scaler.mean_ = numpy.array(stats_dict['input_mean'])
            self.input_scaler.scale_ = numpy.array(stats_dict['input_scale'])

        self._X = torch.tensor(self.input_scaler.transform(self._df_ml[predictor_features]),  
                               dtype=torch.float32)


        self.target_encoder = sklearn.preprocessing.LabelBinarizer(sparse_output=False)
        if stats_dict is None:
            self.target_encoder.fit(self._df_ml[[target_feature]])
        else:
            self.target_encoder.classes_ = numpy.array(stats_dict['target_classes'],dtype='object')

        
        self._y = torch.tensor(self.target_encoder.transform(self._df_ml[[target_feature]]),
                               dtype=torch.float32)

        self.stats_dict = {
            'input_mean': [float(v1) for v1 in self.input_scaler.mean_],
            'input_scale': [float(v1) for v1 in self.input_scaler.scale_],
            'target_classes': list(self.target_encoder.classes_),
        }
        
    def _repr_html_(self):
        return f'''
        <h1>Climate Zones Dataset</h1>
        Number of samples {len(self._X)}
        '''
    
    def __len__(self):
        return len(self._X)

    def __getitem__(self,idx):
        return self._X[idx], self._y[idx]


        

We now intialise the validate and test set data loaders. Note that we initialise the preprocessing objects with the values learned from the training data, rather than calculating them on the validate or test data.

### Using our dataset class
Once we have defined the class, we can now initialise objects from it. A typical pattern is to define separate objects for the train, validate and test sets. You then define an iterator, i.e. a data loader object, for each of the dataset objects. Data loaders are a generic class defined by PyTorch that should work with any dataset that complies with the standard interface.

In [37]:
cz_train_ds = ClimateZonesDataset(train_df, predictors, target_var, device)
cz_train_ds

In [38]:
cz_val_ds = ClimateZonesDataset(val_df, predictors, target_var, device, cz_train_ds.stats_dict)
cz_test_ds = ClimateZonesDataset(test_df, predictors, target_var, device, cz_train_ds.stats_dict)

/gws/ssde/j25b/mohc_shared/users/shaddad/venv/dscop_torch_nb_gpu/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/gws/ssde/j25b/mohc_shared/users/shaddad/venv/dscop_torch_nb_gpu/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [39]:
len(cz_train_ds)

325206

In [40]:
cz_train_loader = torch.utils.data.DataLoader(
        cz_train_ds, batch_size=training_params['batch_size'], shuffle=True, num_workers=1,
    )
cz_val_loader = torch.utils.data.DataLoader(
        cz_val_ds, batch_size=training_params['batch_size'], shuffle=False, num_workers=1,
    )

In [41]:
training_params['batch_size']

16

In [43]:
next(iter(cz_train_loader))

[tensor([[ 6.8058e-01,  6.4744e-01,  1.4899e+00,  2.1942e+00,  1.3117e+00,
           1.8420e-01, -3.2380e-01, -4.4525e-02,  3.2903e-01,  6.4858e-01,
           1.6029e+00,  1.2033e+00,  1.3552e+00,  1.3086e+00,  1.1624e+00,
           9.9240e-01,  8.4752e-01,  7.2524e-01,  6.6307e-01,  7.0645e-01,
           8.2323e-01,  1.0138e+00,  1.2118e+00,  1.3315e+00],
         [-5.9971e-01, -6.1016e-01, -6.5336e-01, -6.8306e-01, -7.1554e-01,
          -6.8584e-01, -6.8380e-01, -5.9377e-01, -7.6682e-01, -7.3253e-01,
          -6.8596e-01, -6.2548e-01, -1.3018e+00, -1.5504e+00, -1.8897e+00,
          -2.0494e+00, -2.1311e+00, -2.1275e+00, -2.1191e+00, -2.1232e+00,
          -2.1161e+00, -2.0277e+00, -1.7145e+00, -1.3632e+00],
         [-5.3210e-01, -4.3329e-01, -5.1154e-02,  1.9796e+00,  3.5675e+00,
           4.8428e+00,  3.8545e+00,  3.3764e+00,  2.8745e+00,  2.5219e+00,
           1.2224e+00, -1.7756e-01,  1.6460e+00,  1.5853e+00,  1.4149e+00,
           1.1940e+00,  1.0067e+00,  8.6283e-01, 

In [44]:
count = 0
for X1,y1 in cz_train_loader:
    print(X1, y1)
    count +=1
    if count > 5:
        break

tensor([[-0.4611, -0.4715, -0.3789, -0.3392, -0.2829, -0.4889, -0.5089, -0.5233,
         -0.4999, -0.5415, -0.4996, -0.5124, -0.5457, -0.6925, -0.8257, -1.0141,
         -1.1228, -1.2031, -1.2475, -1.2310, -1.1970, -1.0888, -0.8672, -0.6292],
        [ 0.1110,  0.1643,  0.1398,  0.2398,  0.8091,  0.6831,  0.4981,  0.2963,
          0.4286,  0.2958,  0.2908,  0.2500,  0.2532,  0.3482,  0.4417,  0.5297,
          0.5733,  0.5963,  0.6259,  0.6423,  0.5566,  0.4979,  0.4137,  0.2524],
        [-0.4155, -0.2946, -0.0760,  0.6274,  0.8576,  0.6237,  0.1779,  0.1910,
          0.2322,  0.0677, -0.1858, -0.3507,  0.0940,  0.2569,  0.4686,  0.5732,
          0.6971,  0.7983,  0.8242,  0.8162,  0.7625,  0.6104,  0.3762,  0.1488],
        [ 1.4454,  1.1389,  1.3744,  0.7459, -0.3789, -0.6744, -0.7171, -0.7712,
         -0.7324, -0.5356,  0.2604,  1.4304,  1.3981,  1.3335,  1.1894,  1.0268,
          0.8785,  0.7338,  0.6548,  0.7127,  0.8601,  1.0872,  1.3217,  1.4131],
        [-0.3166, -0.266

## Building a pytorch model
The next step is build a class to encapsulate the architecture of the ML model that we are going to train. As with the dataset, we do this by


### Further reading
- [Classification tutorial - Pytorch docs](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html)
- [Classification tutorial - Machine Learning Mastery](https://machinelearningmastery.com/building-a-multiclass-classification-model-in-pytorch/ )

In [92]:
class ClimateZoneClassifier(lightning.LightningModule):
    def __init__(self):
        super().__init__()
        self._create_model()
        self._loss_fn = torch.nn.CrossEntropyLoss()

    def _create_model(self):
        self._model = torch.nn.Sequential(
        torch.nn.Linear(24, 60),
        torch.nn.ReLU(),
        torch.nn.Linear(60, 60),
        torch.nn.ReLU(),
        torch.nn.Linear(60, 60),
        torch.nn.ReLU(),
        torch.nn.Linear(60, 5),
        # torch.nn.Sigmoid(),
        torch.nn.Softmax(dim=-1)            ,
        )
        
    def forward(self, x):
        x = self._model(x) 
        return x

    # def forward(self, inputs, target):
    #     return self._model(inputs, target)
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        y_pred = self(x)
        batch_train_loss = self._loss_fn(y_pred, y)
        self.log('train_loss', batch_train_loss)
        return batch_train_loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_pred = self(x)
        batch_val_loss = self._loss_fn(y_pred, y)
        self.log('val_loss', batch_val_loss)

    def test_step(self, batch, batch_idx):
        x, y = batch
        y_pred = self(x)
        batch_test_loss = self._loss_fn(y_pred, y)
        self.log('test_loss', batch_test_loss)

    def predict_step(self, batch, batch_idx):
        x, y = batch
        y_pred = self(x)
        return pred        
        
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), 
                                     training_params['learning_rate']
                                    )
        return optimizer
    


In [93]:
cz_classifier = ClimateZoneClassifier()

In [94]:
cz_classifier

ClimateZoneClassifier(
  (_model): Sequential(
    (0): Linear(in_features=24, out_features=60, bias=True)
    (1): ReLU()
    (2): Linear(in_features=60, out_features=60, bias=True)
    (3): ReLU()
    (4): Linear(in_features=60, out_features=60, bias=True)
    (5): ReLU()
    (6): Linear(in_features=60, out_features=5, bias=True)
    (7): Softmax(dim=-1)
  )
  (_loss_fn): CrossEntropyLoss()
)

### Experiment tracking with mlflow
Next we set up experiment tracking as in the first Pytorch tutorial.

In [50]:
import mlflow

In [51]:
try:
    mlflow_port = os.environ['MLFLOW_PORT']
except KeyError:
    mlflow_port = 4455    
mlflow_server_uri = f'http://localhost:{mlflow_port}'


In [52]:
print(f'connecting to mlflow server {mlflow_server_uri}')
mlflow.set_tracking_uri(mlflow_server_uri)

connecting to mlflow server http://localhost:4455


In [53]:
mlflow.pytorch.autolog()

In [54]:
exp_name='climate_zones_lightning_nn'

In [55]:
if mlflow.get_experiment_by_name(exp_name) is None:
    exp_id = mlflow.create_experiment(exp_name)
exp1 = mlflow.get_experiment_by_name(exp_name)
exp1

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1788273674310, experiment_id='1', last_update_time=1788273674310, lifecycle_stage='active', name='climate_zones_lightning_nn', tags={}>

In [88]:
cz_signature = mlflow.models.infer_signature(cz_train_ds[:5][0].numpy(), cz_train_ds[:5][1].numpy())

## Run the training loop

In pytorch lightning, our model specifies the steps to use for training, validation, testingand inference, so we simply call the Trainer class to run the training loop.

### Further reading
* [Introduction to lightning](https://lightning.ai/docs/pytorch/stable/home/introduction)
* [Tutorial on converting Pytorch code to Pytorch Lightning](https://lightning.ai/docs/pytorch/stable/reference/starter/converting)
* [Lightning Module Docs](https://lightning.ai/docs/pytorch/stable/core-api/lightning_module)


In [89]:
num_epochs = training_params['num_epochs']
# num_epochs = 1 # for debug
num_epochs

2

In [95]:
trainer = lightning.Trainer(max_epochs=num_epochs)


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [96]:
trainer.fit(
    model=cz_classifier,
    train_dataloaders=cz_train_loader,
    val_dataloaders=cz_val_loader,
)

2026/09/01 16:36:30 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'f93ab6a6df744e80913fb8af637fb80d', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current pytorch workflow
2026/09/01 16:36:30 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/gws/ssde/j25b/mohc_shared/users/shaddad/venv/dscop_torch_nb_gpu/lib/python3.12/site-packages/mlflow/pytorch/_lightning_autolog.py:542: UserWarning: Autologging is known to be compatible with pytorch-lightning versions between 2.1.4 and 2.6.0 and may not succeed with packages outside this range."
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type             | Params | Mode

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/gws/ssde/j25b/mohc_shared/users/shaddad/venv/dscop_torch_nb_gpu/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/gws/ssde/j25b/mohc_shared/users/shaddad/venv/dscop_torch_nb_gpu/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=63` in the `DataLoader` to improve performance.


/gws/ssde/j25b/mohc_shared/users/shaddad/venv/dscop_torch_nb_gpu/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=63` in the `DataLoader` to improve performance.


Epoch 0: 100%|██████████| 20326/20326 [01:23<00:00, 243.90it/s, v_num=7]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0:  39%|███▉      | 997/2541 [00:02<00:04, 344.38it/s]

...


Epoch 1: 100%|██████████| 20326/20326 [01:30<00:00, 224.66it/s, v_num=7]     

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=2` reached.


Epoch 1: 100%|██████████| 20326/20326 [01:31<00:00, 221.96it/s, v_num=7]


2026/09/01 16:39:34 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


🏃 View run tasteful-shrew-905 at: http://localhost:4455/#/experiments/0/runs/f93ab6a6df744e80913fb8af637fb80d
🧪 View experiment at: http://localhost:4455/#/experiments/0


In [ ]:
def get_classification_metrics(model, cz_data, set_label, target_encoder):
    class_labels = target_encoder.classes_
    return pandas.DataFrame({
        'climate_group': class_labels,
        f'precision_{set_label}': sklearn.metrics.precision_score(
            target_encoder.inverse_transform(model(cz_data._X.to(device)).to('cpu').detach().numpy()),
            target_encoder.inverse_transform(cz_data._y.numpy()),
            average=None,
            labels=class_labels,
        ),
        f'recall_{set_label}': sklearn.metrics.recall_score(
            target_encoder.inverse_transform(model(cz_data._X.to(device)).to('cpu').detach().numpy()),
            target_encoder.inverse_transform(cz_data._y.numpy()),
            average=None,
            labels=class_labels,
        ),
    })

In [ ]:
%%time
with mlflow.start_run(experiment_id=exp1.experiment_id) as current_run:
    print(current_run.info.run_id)
    mlflow.log_params(training_params)    
    mlflow.log_dict(cz_train_ds.stats_dict, 'stats.json')
    for epoch in range(num_epochs):
        print(f'epoch {epoch}')
        cz_classifier.train()
        epoch_loss_train = 0.0
        for batch_X, batch_y in cz_train_loader:
            optimizer.zero_grad()
            predictions = cz_classifier(batch_X.to(device))
            loss = loss_fn(predictions, batch_y.to(device))
            loss.backward()
            optimizer.step()
            epoch_loss_train += loss.to('cpu').item()

        #divide by number of batches
        epoch_loss_train /= len(cz_train_loader)
        
        epoch_loss_val = 0.0
        for X_val, y_val in cz_val_loader:
            epoch_loss_val += loss_fn(cz_classifier(X_val.to(device)), y_val.to(device)).item()    
        epoch_loss_val /= len(cz_val_loader)

        mlflow.log_metrics(
            { 'cross_entropy_train': epoch_loss_train,
            'cross_entropy_val': epoch_loss_val, },
            step=epoch,
        )
            
    metrics_df = get_classification_metrics(cz_classifier,
                                               cz_train_ds, 
                                               'train', 
                                               target_encoder=cz_train_ds.target_encoder)
    
    metrics_df = metrics_df.merge( get_classification_metrics(cz_classifier,
                                               cz_val_ds, 
                                               'val', 
                                               target_encoder=cz_train_ds.target_encoder), on='climate_group')
    mlflow.log_table(metrics_df, 'metrics.json')


In [ ]:
torch.save(cz_classifier,'cz_model.pth')
mlflow.log_artifact('cz_model.pth')


### Load model and do inference

After the run, we can look at the experiment to see details of the run. On some systems we can use a GUI to inspect results, by going to the same URI at the server, but on many systems the GUI is not available, so we can access the results through the python interface. Here we search through runs in an experiment, and look up a metric for the recent run.

In [ ]:
mlflow.search_runs(exp1.experiment_id)

In [ ]:
model_inference = torch.load('cz_model.pth', weights_only=False)


In [ ]:
model_inference

In [ ]:
cz_train_ds.target_encoder.inverse_transform(
    model_inference(cz_val_ds._X.to(device)).to('cpu').detach().numpy()
)
        

### Evaluation

We will go into more details on evaluating the performance of a machine learning model in a separate notebook.

In [ ]:
# load metrics from mlflow to see training

In [ ]:
current_run.info.run_id

In [ ]:
metrics_path = pathlib.Path(mlflow.artifacts.download_artifacts(f'runs:/{current_run.info.run_id}/metrics.json'))
metrics_path

In [ ]:
with open(metrics_path) as metrics_file:
    metrics_mlflow_dict = json.load(metrics_file)
metrics_mlflow_dict    

In [ ]:
pandas.DataFrame(metrics_mlflow_dict['data'],columns=metrics_mlflow_dict['columns'])

In [ ]:
mlflow.search_runs(exp1.experiment_id,
                   order_by=['metrics.cross_entropy_val'],
                   max_results=5,
                  )

In [ ]:
# calculate metrics from predictions on test vset

In [ ]:
test_metrics_df = get_classification_metrics(cz_classifier,
                                             cz_test_ds, 
                                             'test', 
                                             target_encoder=cz_train_ds.target_encoder)
test_metrics_df


# Exercises

Once you have worked through the tutorial above, you can test your knowledge by adapting the code in the tutorial to experiment with different options for training the model and comparing results.

### Varying model architecture
Try to change the model architecture e.g. add more layers

In [ ]:
# insert code here

### Next steps or potential follow on material

Links from notebook
- [PyTorch docs](https://pytorch.org/)

Additional excercises in this tutorial material includes:
- Training a CNN autoencoder on CMIP6 data
- Training a RNN on Jena weather dataset for time series prediction. 


### Data statement
This data used in this notebook is derived from the Koppen-Geiger Climate Cliassfication dataset created by GloH2O.

###     References
- [Climate Zones Dataset](https://www.gloh2o.org/koppen/#:~:text=The%20K%C3%B6ppen%2DGeiger%20climate%20classification%20maps%20are%20high%2Dresolution,Climate%20Sensitivity%20(ECS)%2C%20and%20historical%20warming%20trend)
